In [3]:
import votekit.ballot_generator as bg
import pickle
from votekit.pref_interval import PreferenceInterval


In [4]:
num_ballots = 10000
# n>6 does not allow for exact sampling of BT, DV
for n in range(2,6):
    print(n)
    n_cands_A = n
    n_cands_B = n
    slate_to_candidates = {"A": [f"A{i}" for i in range(n_cands_A)],
                            "B": [f"B{i}" for i in range(n_cands_B)]}
    
    for pi in [.5, .6, .7, .8, .9, .99]:
        print(pi)
        cohesion = {"B":{"A": 1-pi, "B":pi},
                        "A":{"B": .8, "A":.2}}

        # just examining 1 bloc at a time
        bloc_props = {"B": 1, "A": 0}

        # for pi_type in ["uniform", "strong", "mixed", "Y"]:
        for pi_type in ["Y"]:
            
            # BA, BB uniform
            if pi_type == "uniform":
                pref_intervals_by_bloc = {"A": {"A": PreferenceInterval({f"A{i}": 1/n_cands_A for i in range(n_cands_A)}),
                                                "B": PreferenceInterval({f"B{i}": 1/n_cands_B for i in range(n_cands_B)})},
                                        "B": {"A": PreferenceInterval({f"A{i}": 1/n_cands_A for i in range(n_cands_A)}),
                                                "B": PreferenceInterval({f"B{i}": 1/n_cands_B for i in range(n_cands_B)})}}
            # BA uniform, BB strong
            elif pi_type == "mixed":
                pref_intervals_by_bloc = {"A": {"A": PreferenceInterval({f"A{i}": 1/n_cands_A for i in range(n_cands_A)}),
                                                "B": PreferenceInterval({f"B{i}": 1/n_cands_B for i in range(n_cands_B)})},
                                        "B": {"A": PreferenceInterval({f"A{i}": 1/n_cands_A for i in range(n_cands_A)}),
                                                "B": PreferenceInterval({f"B{i}": 10/(n_cands_B+9) if i == 0 else 1 / (n_cands_B + 9) for i in range(n_cands_B)})}}
            # BA strong, BB uniform
            elif pi_type == "Y":
                pref_intervals_by_bloc = {"A": {"A": PreferenceInterval({f"A{i}": 1/n_cands_A for i in range(n_cands_A)}),
                                                "B": PreferenceInterval({f"B{i}": 1/n_cands_B for i in range(n_cands_B)})},
                                        "B": {"A": PreferenceInterval({f"A{i}": 10/(n_cands_A+9) if i == 0 else 1/(n_cands_A+9) for i in range(n_cands_A)}),
                                                "B": PreferenceInterval({f"B{i}": 1/(n_cands_B)for i in range(n_cands_B)})}}
            # BA, BB strong
            else:
                pref_intervals_by_bloc = {"A": {"A": PreferenceInterval({f"A{i}": 10/(n_cands_A+9) if i == 0 else 1 / (n_cands_A + 9) for i in range(n_cands_A)}),
                                                "B": PreferenceInterval({f"B{i}": 10/(n_cands_B+9) if i == 0 else 1 / (n_cands_B + 9) for i in range(n_cands_B)})},
                                        "B": {"A": PreferenceInterval({f"A{i}": 10/(n_cands_A+9) if i == 0 else 1 / (n_cands_A + 9) for i in range(n_cands_A)}),
                                                "B": PreferenceInterval({f"B{i}": 10/(n_cands_B+9) if i == 0 else 1 / (n_cands_B + 9) for i in range(n_cands_B)})}}

            

            for g in ["BT_name", "PL_name", "PL_type", "BT_type", "CS B=C", "CS B=W"]:
            # for g in ["BS"]:
            # for g in ["BradleyTerry"]:
                # print(g)
                if g == "BT_name":
                    generator = bg.BradleyTerry(candidates = slate_to_candidates["A"] + slate_to_candidates["B"],
                                        bloc_voter_prop = bloc_props,
                                        cohesion_parameters=cohesion,
                                        pref_intervals_by_bloc = pref_intervals_by_bloc,
                                        slate_to_candidates=slate_to_candidates)
                
                elif g == "PL_name":
                    generator = bg.PlackettLuce(candidates = slate_to_candidates["A"] + slate_to_candidates["B"],
                                        bloc_voter_prop = bloc_props,
                                        cohesion_parameters=cohesion,
                                        pref_intervals_by_bloc = pref_intervals_by_bloc,
                                        slate_to_candidates=slate_to_candidates)
                
                    
                    
                elif g == "PL_type":
                    generator = bg.SlatePreference(candidates = slate_to_candidates["A"] + slate_to_candidates["B"],
                                        bloc_voter_prop = bloc_props,
                                        cohesion_parameters=cohesion,
                                        pref_intervals_by_bloc = pref_intervals_by_bloc,
                                        slate_to_candidates=slate_to_candidates)
                    
                elif g == "CS B=C":
                    # just examining 1 bloc at a time
                    bloc_props = {"B": .25, "A": .75}
                    generator = bg.CambridgeSampler(candidates = slate_to_candidates["A"] + slate_to_candidates["B"],
                                        bloc_voter_prop = bloc_props,
                                        cohesion_parameters=cohesion,
                                        pref_intervals_by_bloc = pref_intervals_by_bloc,
                                        slate_to_candidates=slate_to_candidates)
                    
                elif g == "CS B=W":
                    # just examining 1 bloc at a time
                    bloc_props = {"B": 1, "A": 0}
                    generator = bg.CambridgeSampler(candidates = slate_to_candidates["A"] + slate_to_candidates["B"],
                                        bloc_voter_prop = bloc_props,
                                        cohesion_parameters=cohesion,
                                        pref_intervals_by_bloc = pref_intervals_by_bloc,
                                        slate_to_candidates=slate_to_candidates)
                    
                else:
                    generator = bg.DeliberativeVoter(candidates = slate_to_candidates["A"] + slate_to_candidates["B"],
                                        bloc_voter_prop = bloc_props,
                                        cohesion_parameters=cohesion,
                                        pref_intervals_by_bloc = pref_intervals_by_bloc,
                                        slate_to_candidates=slate_to_candidates)

                
                if g  == "CS B=W":
                    pp_dict, pp_agg = generator.generate_profile(number_of_ballots=num_ballots, by_bloc=True)
                    pp = pp_dict["W"]
                
                elif g ==  "CS B=C":
                     pp_dict, pp_agg = generator.generate_profile(number_of_ballots=4*num_ballots, by_bloc=True)
                     pp = pp_dict["C"]
                else:
                    pp = generator.generate_profile(number_of_ballots=num_ballots)
                
                with open(f'ranked_marginals_profiles/{g}_{pi_type}_n_{n}_pi_{pi}.pkl', 'wb') as file:
                    pickle.dump(pp.to_dict(), file)

2
0.5
0.6
0.7
0.8
0.9
0.99
3
0.5
0.6
0.7
0.8
0.9
0.99
4
0.5
0.6
0.7
0.8
0.9
0.99
5
0.5
0.6
0.7
0.8
0.9
0.99
